# Consistency Evaluations

Analysis of the per-checkpoint `consistency_evals.csv` logged during training.

**Walk-forward note:** each fold writes its own log. The loader below auto-follows
the most-recently-updated one (the fold training right now); set `FOLD` in that cell
to pin a specific walk-forward fold (`1`, `2`, …) or the final-fold log (`"production"`).

The main chart compares `train_eval_r` and `val_r` against PPO training timesteps;
the green line marks the best-scoring (drawdown-penalised, both-legs-profitable) checkpoint.

In [121]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

# ── Locate the consistency log ───────────────────────────────────────────────
# Walk-forward training writes ONE log per fold:
#   models/walk_forward/fold_k/eval_logs/consistency_evals.csv   (folds 1 … n-1)
#   models/eval_logs/consistency_evals.csv                       (final / production fold)
# Default = follow the most-recently-updated log, i.e. the fold training right now.
# Override FOLD to pin a specific one:
#   FOLD = None          -> auto (latest modified)
#   FOLD = 1, 2, …       -> that walk-forward fold
#   FOLD = "production"   -> the final-fold log at models/eval_logs/
FOLD = None


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "models").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a 'models' directory from the current working directory.")


def resolve_evals_path(root: Path, fold) -> Path:
    models = root / "models"
    if fold == "production":
        return models / "eval_logs" / "consistency_evals.csv"
    if isinstance(fold, int):
        return models / "walk_forward" / f"fold_{fold}" / "eval_logs" / "consistency_evals.csv"
    # auto: newest consistency_evals.csv anywhere under models/
    candidates = sorted(
        models.glob("**/eval_logs/consistency_evals.csv"),
        key=lambda p: p.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError("No consistency_evals.csv found under models/. Train first.")
    return candidates[-1]


PROJECT_ROOT = find_project_root()
EVALS_PATH = resolve_evals_path(PROJECT_ROOT, FOLD)

df = pd.read_csv(EVALS_PATH).sort_values("timesteps").reset_index(drop=True)
df["timesteps_k"] = df["timesteps"] / 1_000

required = {"timesteps", "train_eval_r", "val_r"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

mtime = pd.Timestamp(EVALS_PATH.stat().st_mtime, unit="s")
print(f"Loaded {len(df)} rows from {EVALS_PATH.relative_to(PROJECT_ROOT)}  (modified {mtime})")
df

Loaded 20 rows from models\sliding\fold_35\eval_logs\consistency_evals.csv  (modified 2026-06-03 21:14:01.210093737)


,timesteps,train_eval_r,val_r,train_dd_pct,val_dd_pct,q_train,q_val,score,gap,eligible,timesteps_k
0,150000,18.579,3.675,5.396,13.257,14.262,-6.930,-6.930,14.904,True,150.0
1,300000,18.435,9.289,8.658,12.459,11.509,-0.678,-0.678,9.146,True,300.0
2,450000,19.928,35.364,9.125,9.434,12.628,27.817,12.628,15.436,True,450.0
3,600000,31.274,-8.635,6.018,13.177,26.460,-19.177,-19.177,39.909,False,600.0
4,750000,34.776,10.994,4.945,10.265,30.820,2.782,2.782,23.782,True,750.0
5,900000,16.670,43.108,10.761,5.684,8.061,38.561,8.061,26.438,True,900.0
6,1050000,4.758,2.984,14.320,11.505,-6.698,-6.220,-6.698,1.774,True,1050.0
7,1200000,20.332,26.243,7.532,8.611,14.307,19.354,14.307,5.910,True,1200.0
8,1350000,0.827,9.178,12.254,9.160,-8.977,1.851,-8.977,8.352,True,1350.0
9,1500000,29.295,18.983,6.975,7.054,23.715,13.340,13.340,10.312,True,1500.0


## Train Eval vs Validation Reward

In [122]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["timesteps"],
        y=df["train_eval_r"],
        mode="lines+markers",
        name="Train eval reward",
        line=dict(width=3),
        hovertemplate="Timesteps: %{x:,}<br>Train eval R: %{y:.3f}<extra></extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=df["timesteps"],
        y=df["val_r"],
        mode="lines+markers",
        name="Validation reward",
        line=dict(width=3),
        hovertemplate="Timesteps: %{x:,}<br>Val R: %{y:.3f}<extra></extra>",
    )
)

if {"train_dd_pct", "val_dd_pct"}.issubset(df.columns):
    fig.add_trace(
        go.Scatter(
            x=df["timesteps"],
            y=df["train_dd_pct"],
            mode="lines+markers",
            name="Train DD %",
            yaxis="y2",
            line=dict(width=2, dash="dot"),
            hovertemplate="Timesteps: %{x:,}<br>Train DD: %{y:.2f}%<extra></extra>",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df["timesteps"],
            y=df["val_dd_pct"],
            mode="lines+markers",
            name="Validation DD %",
            yaxis="y2",
            line=dict(width=2, dash="dash"),
            hovertemplate="Timesteps: %{x:,}<br>Val DD: %{y:.2f}%<extra></extra>",
        )
    )

if "score" in df.columns and df["score"].notna().any():
    best_idx = df["score"].idxmax()
    best = df.loc[best_idx]
    fig.add_vline(
        x=best["timesteps"],
        line_width=2,
        line_dash="dash",
        line_color="green",
        annotation_text=f"Best score: {int(best['timesteps']):,}",
        annotation_position="top left",
    )

fig.update_layout(
    title="Train Eval Reward and Validation Reward vs Timesteps",
    template="plotly_white",
    width=1000,
    height=540,
    hovermode="x unified",
    xaxis_title="Training timesteps",
    yaxis_title="Episode reward",
    yaxis2=dict(
        title="Drawdown %",
        overlaying="y",
        side="right",
        rangemode="tozero",
        showgrid=False,
    ),
    legend_title="Series",
)
fig.update_xaxes(tickformat=",")

fig.show()

## Best Checkpoint Summary

In [123]:
if "score" in df.columns and df["score"].notna().any():
    best = df.loc[df["score"].idxmax()].copy()
else:
    best = df.loc[df["val_r"].idxmax()].copy()

summary_cols = [
    "timesteps",
    "train_eval_r",
    "val_r",
    "train_dd_pct",
    "val_dd_pct",
    "q_train",
    "q_val",
    "score",
    "gap",
    "eligible",
]
summary_cols = [col for col in summary_cols if col in best.index]

best[summary_cols].to_frame("best_checkpoint")

,best_checkpoint
timesteps,2250000
train_eval_r,52.602
val_r,35.702
train_dd_pct,5.171
val_dd_pct,8.407
q_train,48.466
q_val,28.976
score,28.976
gap,16.901
eligible,True


## Full Table

In [124]:
display_cols = [
    "timesteps",
    "train_eval_r",
    "val_r",
    "train_dd_pct",
    "val_dd_pct",
    "score",
    "gap",
    "eligible",
]
display_cols = [col for col in display_cols if col in df.columns]
df[display_cols]

,timesteps,train_eval_r,val_r,train_dd_pct,val_dd_pct,score,gap,eligible
0,150000,18.579,3.675,5.396,13.257,-6.930,14.904,True
1,300000,18.435,9.289,8.658,12.459,-0.678,9.146,True
2,450000,19.928,35.364,9.125,9.434,12.628,15.436,True
3,600000,31.274,-8.635,6.018,13.177,-19.177,39.909,False
4,750000,34.776,10.994,4.945,10.265,2.782,23.782,True
5,900000,16.670,43.108,10.761,5.684,8.061,26.438,True
6,1050000,4.758,2.984,14.320,11.505,-6.698,1.774,True
7,1200000,20.332,26.243,7.532,8.611,14.307,5.910,True
8,1350000,0.827,9.178,12.254,9.160,-8.977,8.352,True
9,1500000,29.295,18.983,6.975,7.054,13.340,10.312,True
